Diffusers and pipeline Code

In [1]:
from diffusers import StableDiffusion3Pipeline
import torch
from config import LAM_VALUES, TSR_DIR, PT_TSR_DIR, PROMPTS_FILE, MODEL_CACHE, SEED, TSR_SIGMA, SWAP_ALGORITHM, N_INF_STEPS, GUIDANCE_SCALE

    # ── Load model once ───────────────────────────────────────────────────────────
pipe = StableDiffusion3Pipeline.from_pretrained(
	"stabilityai/stable-diffusion-3-medium-diffusers",
	torch_dtype=torch.float16,
	cache_dir=MODEL_CACHE,
)
pipe = pipe.to("cuda")
pipe.set_progress_bar_config(disable=True)

Loading pipeline components...:   0%|          | 0/9 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Imports, configs

In [2]:
from pathlib import Path
import gc
import pandas as pd
from tqdm import tqdm
import shutil

TSR_DIR     = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/tsr_samples_tester_new")
PT_TSR_DIR  = Path("/n/netscratch/kempner_undergrads/Everyone/zwu/parallel_toy/images/pt_samples_tester_new")

REPLICA_EXCHANGE = True
LAM_VALUES = [1.1]
INDEX_UNTIL = 6

gc.collect()
torch.cuda.empty_cache()

SWAP_ALGORITHM = {
	"n_replicas": 2,
	"p_ratio": "p",
	"even_indices": [0,1,  2,3,  4,5,  6,7],
	"odd_indices": 	[],
	"debug": True,
}
    

In [3]:
if PT_TSR_DIR.exists():
    shutil.rmtree(PT_TSR_DIR)


Sample!

In [ ]:
# ── Load prompts once ─────────────────────────────────────────────────────────
prompts = pd.read_csv(PROMPTS_FILE, usecols=["text"], nrows=INDEX_UNTIL)["text"].tolist()
print(f"Loaded {len(prompts)} prompts")


replica_exchanges = [False, True]

lam_dirs = {}
for re in replica_exchanges:
	base = PT_TSR_DIR if re else TSR_DIR
	lam_dirs[re] = {l: base / f"lam{l:.3f}".replace(".", "p") for l in LAM_VALUES}
	for d in lam_dirs[re].values():
		d.mkdir(parents=True, exist_ok=True)

# ── Sweep ─────────────────────────────────────────────────────────────────────
for idx, prompt in enumerate(prompts):
	
	for replica_exchange in replica_exchanges:
	
		for tsr_lam in tqdm(LAM_VALUES, desc=f"idx={idx} re={replica_exchange}"):

			output_dir = lam_dirs[replica_exchange][tsr_lam]

			if (output_dir / f"{idx:05d}.png").exists():
				continue

			generator = torch.Generator(device="cuda").manual_seed(SEED)

			if replica_exchange: tsr_sigma = 50
			else: tsr_sigma = 3

			images = pipe(
				prompt,
				negative_prompt="",
				num_inference_steps=N_INF_STEPS,
				guidance_scale=GUIDANCE_SCALE,
				tsr_lam=tsr_lam,
				tsr_sigma=tsr_sigma,
				replica_exchange=replica_exchange,
				swap_algorithm=SWAP_ALGORITHM,
				generator=generator,
			).images

			out_path = output_dir / f"{idx:05d}.png"
			images[0].save(out_path, icc_profile=None)


			del images
		torch.cuda.empty_cache()

print("\n All k values complete.")

Loaded 6 prompts


idx=0 re=False:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.10 with replica exchange False


idx=0 re=True:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.10 with replica exchange True
Time 1000.00 swap btwn source 0.30 and target 0.91 accept 0.999 std 0.989
Time 988.27 swap btwn source 0.30 and target 0.91 accept 0.992 std 0.981
Time 975.98 swap btwn source 0.30 and target 0.91 accept 0.980 std 0.973
Time 963.08 swap btwn source 0.30 and target 0.91 accept 0.987 std 0.965
Time 949.53 swap btwn source 0.30 and target 0.91 accept 0.977 std 0.957
Time 935.28 swap btwn source 0.30 and target 0.91 accept 0.981 std 0.948
Time 920.28 swap btwn source 0.30 and target 0.91 accept 0.976 std 0.938
Time 904.45 swap btwn source 0.30 and target 0.91 accept 0.978 std 0.929


...

 We tsr by 1.10 with replica exchange True
Time 1000.00 swap btwn source 0.30 and target 0.91 accept 0.999 std 0.990
Time 988.27 swap btwn source 0.30 and target 0.91 accept 0.984 std 0.982
Time 975.98 swap btwn source 0.30 and target 0.91 accept 0.995 std 0.975
Time 963.08 swap btwn source 0.30 and target 0.91 accept 0.982 std 0.967
Time 949.53 swap btwn source 0.30 and target 0.91 accept 0.992 std 0.958
Time 935.28 swap btwn source 0.30 and target 0.91 accept 0.980 std 0.950
Time 920.28 swap btwn source 0.30 and target 0.91 accept 0.987 std 0.940
Time 904.45 swap btwn source 0.30 and target 0.91 accept 0.978 std 0.931


idx=3 re=False:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.10 with replica exchange False


idx=3 re=True:   0%|          | 0/1 [00:00<?, ?it/s]

 We tsr by 1.10 with replica exchange True
Time 1000.00 swap btwn source 0.30 and target 0.91 accept 1.000 std 0.990
Time 988.27 swap btwn source 0.30 and target 0.91 accept 0.995 std 0.982
Time 975.98 swap btwn source 0.30 and target 0.91 accept 0.998 std 0.975
Time 963.08 swap btwn source 0.30 and target 0.91 accept 0.989 std 0.967
Time 949.53 swap btwn source 0.30 and target 0.91 accept 0.993 std 0.958
Time 935.28 swap btwn source 0.30 and target 0.91 accept 0.984 std 0.950
Time 920.28 swap btwn source 0.30 and target 0.91 accept 0.987 std 0.941
Time 904.45 swap btwn source 0.30 and target 0.91 accept 0.979 std 0.931


Fid computation

In [ ]:
from fid import compute_sweep
from config import LAM_VALUES, PT_TSR_DIR, TSR_DIR
compute_sweep(
	lam_values=LAM_VALUES,
	replica_exchanges=[True, False],
	device="cuda",
	target_indices=None,
	index_until = 28,
	pt_sr_dir = PT_TSR_DIR,
	tsr_dir = TSR_DIR,
)